# Chapter 10 — Evaluation & CI for LLM Pipelines (v2026)

> **LangChain 1.x / 2026 refresh.** Dual-mode secrets, optional LangSmith tracing, pinned core deps where applicable, and a standardized footer (Limitations & safety + cleanup + exercises). **REFACTORED_EVALUATION_CI_V2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/CHDIR/FNAME)

## Learning objectives
- Define benchmark fixtures with expected outputs
- Score a pipeline against fixtures
- Enforce regression thresholds as CI-ready asserts

> **Runtime / cost / data.** Offline; deterministic fixtures and asserts.

## Environment setup

In [ ]:
import os

# Secrets are read from Colab Secrets if available, else from a local .env
try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)

OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Optional LangSmith tracing (set LANGCHAIN_API_KEY to enable)
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter10-evaluation-ci"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")

## Why CI for LLM pipelines?

Model and dependency updates silently change behavior. A **regression suite** with hard thresholds turns 'it seems worse' into a failing assert that blocks a release.

We: (1) define fixtures, (2) run a toy pipeline, (3) score accuracy, (4) assert a threshold.

In [ ]:
# 1) Benchmark fixtures (question -> expected keyword)
FIXTURES = [
    {"q": "First-line drug for type 2 diabetes?", "expect": "metformin"},
    {"q": "Which class reduces cardiovascular events?", "expect": "sglt2"},
    {"q": "Which class supports weight loss?", "expect": "glp-1"},
    {"q": "Standard first-line for hypertension?", "expect": "ace"},
]
len(FIXTURES)

In [ ]:
# 2) Toy pipeline (keyword retrieval over a mini knowledge base)
KB = {
    "metformin": "Metformin is first-line for type 2 diabetes.",
    "sglt2": "SGLT2 inhibitors reduce cardiovascular events.",
    "glp-1": "GLP-1 agonists support weight reduction.",
    "ace": "ACE inhibitors are a standard first-line for hypertension.",
}

def pipeline(q):
    ql = q.lower()
    for k, v in KB.items():
        if k in ql or any(t in ql for t in k.split()):
            return v
    # fallback: pick by keyword overlap
    best = max(KB.values(), key=lambda v: len(set(ql.split()) & set(v.lower().split())))
    return best

print(pipeline("First-line drug for type 2 diabetes?"))

In [ ]:
# 3) Score accuracy against fixtures
def evaluate(fixtures):
    results = []
    for f in fixtures:
        out = pipeline(f["q"]).lower()
        ok = f["expect"] in out
        results.append({"q": f["q"], "expect": f["expect"], "pass": ok})
    acc = sum(r["pass"] for r in results) / len(results)
    return acc, results

acc, results = evaluate(FIXTURES)
print(f"accuracy = {acc:.2f}")
results

In [ ]:
# 4) CI-ready regression threshold
THRESHOLD = 0.75
report = {"accuracy": acc, "threshold": THRESHOLD, "n": len(FIXTURES)}
print(report)
assert acc >= THRESHOLD, f"REGRESSION: accuracy {acc:.2f} < {THRESHOLD}"
print("CI gate PASSED")

## Going further

Real suites add: semantic similarity scorers, LLM-as-judge (with a fixed rubric), per-category thresholds, and golden-dataset versioning wired into `pytest`.

## Limitations & safety

- **Enterprise / research-support only.** Human review is required before any production, clinical, or compliance decision.
- **Synthetic / de-identified data only.** Real PHI/PII requires governance and access controls.

In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model", "agent", "app", "manifest", "report"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")

## Exercises

<details><summary>Q1. Why is a run manifest essential for reproducibility in an enterprise pipeline?</summary>
It captures corpus/index versions, prompts, and model/tool versions, so a past result can be audited, diffed, or re-run deterministically when models or data change.
</details>

<details><summary>Q2. Why treat guardrail / injection detections as drafts rather than automatic enforcement?</summary>
Detectors have false positives and negatives. In regulated settings, an error can block legitimate work or leak PHI, so a human confirms before enforcement.
</details>

<details><summary>Q3. Why compare frameworks by task and operations needs instead of popularity?</summary>
The best fit depends on state management, observability, deployment, and team skill — not download counts. A task-driven matrix makes the trade-offs explicit and defensible.
</details>

### Task A — Split fixtures by category (drug-class vs condition) and assert a per-category threshold.

### Task B — Add a latency budget: time the pipeline and assert p95 latency is under a limit.

### Task C — Serialize the results to JSON and add a `--ci` flag that exits non-zero on failure.

### Task D — Add a semantic scorer (e.g., token-overlap F1) alongside the keyword check and compare pass rates.